In [4]:
from datetime import datetime
from pymongo import MongoClient
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager
import time

# ==========================================================
# CONEXION MONGODB
# ==========================================================
try:
    client = MongoClient("mongodb://bigdata_mongodb3:27017/")
    db = client["proyecto_bigdata"]
    coleccion = db["alojamientos"]
    print("Conexion a MongoDB exitosa!")
except Exception as e:
    print(f"Error conectando a MongoDB: {e}")

# ==========================================================
# CIUDADES
# ==========================================================
ciudades = [
    "Arica","Iquique","Calama","Antofagasta","Copiapo",
    "La Serena","Valparaiso","Vina del Mar","Santiago",
    "Rancagua","Talca","Chillan","Concepcion","Temuco",
    "Valdivia","Puerto Varas","Puerto Montt","Coyhaique",
    "Puerto Natales","Punta Arenas"
]

# ==========================================================
# FUNCIONES
# ==========================================================
def limpiar_precio(texto):
    numeros = ""
    for c in texto:
        if c.isdigit():
            numeros += c
    return float(numeros) if numeros else 0.0


def configurar_driver():
    opciones = Options()
    opciones.add_argument("--headless")
    opciones.add_argument("--no-sandbox")
    opciones.add_argument("--disable-dev-shm-usage")
    opciones.add_argument("--disable-blink-features=AutomationControlled")
    opciones.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
    )

    return webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=opciones
    )


def obtener_zona(ciudad):
    norte = ["Arica","Iquique","Calama","Antofagasta","Copiapo","La Serena"]
    centro = ["Valparaiso","Vina del Mar","Santiago","Rancagua","Talca"]

    if ciudad in norte:
        return "Norte"
    elif ciudad in centro:
        return "Centro"
    return "Sur"


# ==========================================================
# SCRAPER HOTELES.COM
# ==========================================================
def scraper_hoteles(ciudad):

    url = f"https://www.hoteles.com/Hotel-Search?destination={ciudad.replace(' ', '+')}%2C+Chile"

    print("=" * 50)
    print("Ciudad:", ciudad)
    print("=" * 50)

    driver = None

    try:
        driver = configurar_driver()
        driver.get(url)

        time.sleep(8)

        # Scroll para cargar resultados
        for _ in range(3):
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            time.sleep(2)

        # Selector de hoteles (puede variar)
        alojamientos = driver.find_elements(By.CSS_SELECTOR, "div[data-stid='property-listing']")

        print("Resultados encontrados:", len(alojamientos))

        guardados = 0

        for i, item in enumerate(alojamientos[:30]):

            try:
                # Nombre
                nombre = item.find_element(By.CSS_SELECTOR, "h3").text

                # Precio
                try:
                    precio_text = item.find_element(By.CSS_SELECTOR, "[data-stid='price-lockup-text']").text
                except:
                    precio_text = item.text

                precio = limpiar_precio(precio_text)

                registro = {
                    "nombre_hotel": nombre,
                    "precio_noche": precio,
                    "ciudad": ciudad,
                    "zona_geografica": obtener_zona(ciudad),
                    "plataforma": "hoteles.com",
                    "fecha_captura": datetime.now(),
                    "integrante": "Juan_Salas",
                    "grupo": "G5_Turismo_Hoteleria"
                }

                coleccion.update_one(
                    {
                        "nombre_hotel": nombre,
                        "ciudad": ciudad,
                        "plataforma": "hoteles.com"
                    },
                    {"$set": registro},
                    upsert=True
                )

                guardados += 1
                print(f"[{i+1}] Guardado -> {nombre[:40]}")

            except:
                continue

        print(f"Total guardados en {ciudad}: {guardados}")
        return guardados

    except Exception as e:
        print("Error:", e)
        return 0

    finally:
        if driver:
            driver.quit()


# ==========================================================
# EJECUCION
# ==========================================================
total_antes = coleccion.count_documents({"plataforma": "hoteles.com"})
print("Registros antes:", total_antes)

total_nuevos = 0

for ciudad in ciudades:
    nuevos = scraper_hoteles(ciudad)
    total_nuevos += nuevos

    print("Esperando 10 segundos...\n")
    time.sleep(10)

total_despues = coleccion.count_documents({"plataforma": "hoteles.com"})

print("=" * 50)
print("SCRAPING COMPLETADO")
print("=" * 50)
print("Antes:", total_antes)
print("Ahora:", total_despues)
print("Nuevos:", total_despues - total_antes)
print("=" * 50)

Conexion a MongoDB exitosa!
Registros antes: 0
Ciudad: Arica
Resultados encontrados: 0
Total guardados en Arica: 0
Esperando 10 segundos...

Ciudad: Iquique
Resultados encontrados: 0
Total guardados en Iquique: 0
Esperando 10 segundos...

Ciudad: Calama
Resultados encontrados: 0
Total guardados en Calama: 0
Esperando 10 segundos...

Ciudad: Antofagasta
Resultados encontrados: 0
Total guardados en Antofagasta: 0
Esperando 10 segundos...

Ciudad: Copiapo
Resultados encontrados: 0
Total guardados en Copiapo: 0
Esperando 10 segundos...

Ciudad: La Serena
Resultados encontrados: 0
Total guardados en La Serena: 0
Esperando 10 segundos...

Ciudad: Valparaiso
Resultados encontrados: 0
Total guardados en Valparaiso: 0
Esperando 10 segundos...

Ciudad: Vina del Mar
Resultados encontrados: 0
Total guardados en Vina del Mar: 0
Esperando 10 segundos...

Ciudad: Santiago
Resultados encontrados: 0
Total guardados en Santiago: 0
Esperando 10 segundos...

Ciudad: Rancagua
Resultados encontrados: 0
Tota